In [2]:
# Importa la librería para realizar solicitudes HTTP a páginas web.
import requests
# Importa BeautifulSoup para parsear y navegar el contenido HTML.
from bs4 import BeautifulSoup
# Importa pandas para manipulación tabular de datos.
import pandas as pd
# Importa datetime para registrar marcas de tiempo.
from datetime import datetime

In [3]:
# Define la URL principal del sitio desde el cual se obtendrá la información de tráfico.
url = "https://traficozmg.com/"

In [4]:
# Realiza una petición GET a la URL definida.
response = requests.get(url)
# Imprime el código de estado HTTP para verificar si la solicitud fue exitosa.
print("Status :", response.status_code)

Status : 403


In [5]:
# Convierte el HTML recibido en un objeto parseable con BeautifulSoup.
soup = BeautifulSoup(response.text, "html.parser")
# Muestra el título de la página para validar que se parseó correctamente.
print(soup.title.text)

Just a moment...


In [6]:
# Inicializa una lista vacía donde se almacenarán los registros extraídos.
data = []

# Busca todos los elementos h2 en el HTML, donde están los textos relevantes.
articles = soup.find_all("h2")

# Recorre cada encabezado encontrado para transformarlo en un registro estructurado.
for article in articles:
    # Limpia y obtiene el texto del encabezado actual.
    text = article.get_text(strip=True)
    # Agrega un diccionario con los campos esperados al arreglo de datos.
    data.append({
        # Guarda la fecha y hora actual como marca temporal del scraping.
        "timestamp": datetime.now(),
        # Asigna una etiqueta general de zona/avenida para el origen textual.
        "avenida": "ZMG",
        # Deja la velocidad como nula porque no viene explícita en el texto.
        "velocidad": None,
        # Deja densidad en nulo para calcularla después a partir del texto.
        "densidad": None,
        # Deja detenciones en nulo por falta de dato directo en la fuente.
        "detenciones": None,
        # Guarda el texto extraído como descripción del estado del tráfico.
        "descripcion": text
    })

# Convierte la lista de diccionarios en un DataFrame de pandas.
df_scraped = pd.DataFrame(data)
# Muestra las primeras filas para inspeccionar el resultado del scraping.
df_scraped.head()

""


In [7]:
# Elimina filas duplicadas para evitar registros repetidos en el dataset scrapeado.
df_scraped = df_scraped.drop_duplicates()

# Verifica que exista la columna descripcion antes de filtrar vacíos.
if "descripcion" in df_scraped.columns:
    # Conserva únicamente descripciones con contenido no vacío tras limpiar espacios.
    df_scraped = df_scraped[df_scraped["descripcion"].fillna("").str.strip() != ""]

# Visualiza una muestra para validar la limpieza aplicada.
df_scraped.head()

""


In [8]:
# Define la ruta de salida para guardar el CSV con los datos scrapeados.
SCRAPED_PATH = "../data/raw/scraped_traffic.csv"
# Exporta el DataFrame a CSV sin incluir el índice numérico.
df_scraped.to_csv(SCRAPED_PATH, index=False)
# Confirma en consola que el archivo fue guardado correctamente.
print("Datos scrapeados guardados")

Datos scrapeados guardados


In [9]:
# Define una función para clasificar la densidad en función de palabras clave del texto.
def classify_density(text):
    # Convierte el texto a minúsculas para comparar sin sensibilidad a mayúsculas.
    text = str(text).lower()
    # Asigna densidad alta cuando detecta términos de tráfico pesado.
    if "fuerte" in text or "congestionado" in text:
        return 3
    # Asigna densidad media cuando detecta términos de carga vehicular.
    elif "carga" in text:
        return 2
    # Asigna densidad baja cuando no encuentra palabras de alta/media densidad.
    else:
        return 1

# Verifica que exista la columna descripcion antes de aplicar la clasificación.
if "descripcion" in df_scraped.columns:
    # Genera la columna densidad aplicando la función de clasificación a cada descripción.
    df_scraped["densidad"] = df_scraped["descripcion"].fillna("").apply(classify_density)
else:
    # Crea una columna vacía tipada como entero nullable si no hay descripciones.
    df_scraped["densidad"] = pd.Series(dtype="Int64")

# Muestra las primeras filas con la nueva columna densidad.
df_scraped.head()

,densidad


In [10]:
# Define la ruta del dataset principal en crudo que se va a complementar.
DATA_RAW_PATH = "../data/raw/traffic_data.csv"
# Carga el dataset principal existente desde disco.
df_main = pd.read_csv(DATA_RAW_PATH)
# Concatena el dataset principal con los nuevos registros scrapeados.
df_combined = pd.concat([df_main, df_scraped], ignore_index=True)
# Sobrescribe el CSV original con el dataset combinado.
df_combined.to_csv(DATA_RAW_PATH, index=False)

# Muestra las últimas filas para confirmar que se añadieron nuevos registros.
df_combined.tail()

,timestamp,avenida,velocidad,densidad,detenciones
13,2026-03-17 22:43:12.258683,Lopez Mateos,20.0,3,2.0
14,2026-03-17 22:43:12.273623,Lopez Mateos,15.0,3,2.0
15,2026-03-20 10:55:33.139701,Lopez Mateos,25.0,3,2.0
16,2026-03-20 10:55:33.177410,Lopez Mateos,20.0,3,2.0
17,2026-03-20 10:55:33.192331,Lopez Mateos,15.0,3,2.0
